In [195]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium
import json
import requests

# Trabajar con archivo 'Transportes'

In [28]:
tra = pd.read_csv("data/01-SecretariadeMovilidad/TRANSPORTES_ARANJUEZ_SANTA_CRUZ.csv", encoding="latin-1", sep=";")
tra.columns = tra.columns.str.lower()

In [56]:
# Atributo esarchivolinea, puede ser interesante

columnas_drop = ["nombrearchivo", "idempresa", "idpersona", "esarchivolinea", "diferenciadepasajeros", "ordenregistro",
       "tipocarga", "idpersonausuario", "archivofirmado","fechavencimientofirma", "indfechacreacion", "fecha_carga",
       "ultimoestadopuertas", "presentocambioenpuertas", "ordenregistro","ipmaquina", "codigo_id",
       "id_tipo_ruta", "nombre_ruta", "secuenciarecorrido", "cantidadregistros", "ultimafecharegistro"]

tra = tra.drop(columnas_drop, axis=1)

In [57]:
tra["ultimalatitudregistro"] = pd.to_numeric(tra["ultimalatitudregistro"].str.replace(",", "."))
tra["ultimalongitudregistro"] = pd.to_numeric(tra["ultimalongitudregistro"].str.replace(",", "."))

In [70]:
tra

,empresa_transporte,idvehiculo,placavehiculo,idruta,codigo_ruta,min_registros_ruta,longitudruta,tipo_ruta,fechainiciodespacho,fechafindespacho,recorridofinalizado,fecha_creacion,cantpasajerossuben,cantpasajerosbajan,ultimalongitudregistro,ultimalatitudregistro,distanciarecorrido,recorridoincumplido,distanciaincumplidainicio,distanciaincumplidafin
0,TRANSPORTES ARANJUEZ SANTA CRUZ,2595,WDX630,803,024,33,"8026,96",Directa no centro,23/09/25 21:55:09,23/09/25 22:54:16,N,23/09/2025 21:55:09,0,0,-75.564079,6.288920,3131,S,"2289,21","182,73"
1,TRANSPORTES ARANJUEZ SANTA CRUZ,5265,TPW235,803,024,33,"8026,96",Directa no centro,23/09/25 21:41:04,23/09/25 21:55:55,S,23/09/2025 21:41:04,1,14,-75.562950,6.290425,3141,S,"2085,51","192,06"
2,TRANSPORTES ARANJUEZ SANTA CRUZ,5268,TPW943,529,041 D,70,"16776,67",Directa centro,23/09/25 21:40:31,23/09/25 22:47:11,S,23/09/2025 21:40:31,4,3,-75.551727,6.287636,13179,S,"216,73","480,45"
3,TRANSPORTES ARANJUEZ SANTA CRUZ,5222,EQS782,529,041 D,70,"16776,67",Directa centro,23/09/25 21:20:32,23/09/25 22:32:46,S,23/09/2025 21:20:32,4,3,-75.552109,6.287522,13399,S,"254,01","454,19"
4,TRANSPORTES ARANJUEZ SANTA CRUZ,5271,TPW970,803,024,33,"8026,96",Directa no centro,23/09/25 21:12:57,23/09/25 21:30:28,S,23/09/2025 21:12:57,4,19,-75.564247,6.288287,3332,S,"2350,1","124,81"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1926,TRANSPORTES ARANJUEZ SANTA CRUZ,5290,TPY711,526,023,75,"18004,93",Directa centro,21/09/25 04:40:32,21/09/25 06:03:54,S,21/09/2025 04:40:32,11,12,-75.552368,6.304451,17437,S,0,"46,74"
1927,TRANSPORTES ARANJUEZ SANTA CRUZ,6892,FWK060,525,022,69,"16621,12",Directa centro,21/09/25 04:37:31,21/09/25 05:52:00,S,21/09/2025 04:37:31,8,8,-75.549599,6.292368,16630,S,0,"188,77"
1928,TRANSPORTES ARANJUEZ SANTA CRUZ,7010,FWK874,526,023,75,"18004,93",Directa centro,21/09/25 04:28:55,21/09/25 05:43:30,S,21/09/2025 04:28:55,3,3,-75.552132,6.304982,17217,S,"68,58","51,16"
1929,TRANSPORTES ARANJUEZ SANTA CRUZ,5272,TPW989,525,022,69,"16621,12",Directa centro,21/09/25 04:23:23,21/09/25 05:18:48,S,21/09/2025 04:23:23,3,4,-75.548569,6.291197,16043,S,"176,82","155,53"


In [71]:
placas = ["EQS783",
 "WMP989",
 "FWK874",
 "EQT504",
 "WMR092"]

placas_presentes = tra[tra["placavehiculo"].isin(placas)]["placavehiculo"].unique()

agrupadas = tra.groupby(by="placavehiculo")
for placa in placas_presentes:
    coords = agrupadas.get_group(placa)[["ultimalatitudregistro","ultimalongitudregistro"]].values.tolist()
    print(f"--Bus de placa: {placa}")
    print(agrupadas.get_group(placa)["codigo_ruta"].unique())

    m = folium.Map(location=coords[0], zoom_start=13)
    for coord in coords:
        folium.Marker(
            location = coord,
            icon=folium.Icon(color="red", icon="info-sign"),
            tooltip=""
        ).add_to(m)

    m.save("mapa_"+placa+".html")

--Bus de placa: EQS783
['041 D' '041 I ']
--Bus de placa: WMR092
['041 D' '041 I ']
--Bus de placa: FWK874
['023' '024']


In [7]:
example1 = tra.groupby(("codigo_ruta")).get_group(("022")).sort_values(by=["ultimalongitudregistro","ultimalatitudregistro"])
coords1 = [coord[::-1] for coord in example1[["ultimalongitudregistro", "ultimalatitudregistro"]].values.tolist()]

example2 = tra.groupby(("codigo_ruta")).get_group(("042")).sort_values(by=["ultimalongitudregistro","ultimalatitudregistro"])
coords2 = [coord[::-1] for coord in example2[["ultimalongitudregistro", "ultimalatitudregistro"]].values.tolist()]

In [8]:
# Create a map object centered at a specific location
m = folium.Map(location=coords1[0], zoom_start=13)

# for coord in coords1:
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="blue", icon="info-sign"),
#         tooltip=""
#     ).add_to(m)

for coord in coords2:
    folium.Marker(
        location = coord,
        icon=folium.Icon(color="red", icon="info-sign"),
        tooltip=""
    ).add_to(m)

m.save("mapa.html")


In [9]:
#Convert format
tra['fechainiciodespacho'] = pd.to_datetime(tra['fechainiciodespacho'], format='%d/%m/%y %H:%M:%S')

In [10]:
tra["fechainicio"] = tra.fechainiciodespacho.dt.date
tra["horainicio"] = tra.fechainiciodespacho.dt.hour

In [11]:
tra

,empresa_transporte,idvehiculo,idruta,codigo_ruta,min_registros_ruta,longitudruta,tipo_ruta,fechainiciodespacho,fechafindespacho,recorridofinalizado,...,cantpasajerossuben,cantpasajerosbajan,ultimalongitudregistro,ultimalatitudregistro,distanciarecorrido,recorridoincumplido,distanciaincumplidainicio,distanciaincumplidafin,fechainicio,horainicio
0,TRANSPORTES ARANJUEZ SANTA CRUZ,2595,803,024,33,"8026,96",Directa no centro,2025-09-23 21:55:09,23/09/25 22:54:16,N,...,0,0,-75.564079,6.288920,3131,S,"2289,21","182,73",2025-09-23,21
1,TRANSPORTES ARANJUEZ SANTA CRUZ,5265,803,024,33,"8026,96",Directa no centro,2025-09-23 21:41:04,23/09/25 21:55:55,S,...,1,14,-75.562950,6.290425,3141,S,"2085,51","192,06",2025-09-23,21
2,TRANSPORTES ARANJUEZ SANTA CRUZ,5268,529,041 D,70,"16776,67",Directa centro,2025-09-23 21:40:31,23/09/25 22:47:11,S,...,4,3,-75.551727,6.287636,13179,S,"216,73","480,45",2025-09-23,21
3,TRANSPORTES ARANJUEZ SANTA CRUZ,5222,529,041 D,70,"16776,67",Directa centro,2025-09-23 21:20:32,23/09/25 22:32:46,S,...,4,3,-75.552109,6.287522,13399,S,"254,01","454,19",2025-09-23,21
4,TRANSPORTES ARANJUEZ SANTA CRUZ,5271,803,024,33,"8026,96",Directa no centro,2025-09-23 21:12:57,23/09/25 21:30:28,S,...,4,19,-75.564247,6.288287,3332,S,"2350,1","124,81",2025-09-23,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1926,TRANSPORTES ARANJUEZ SANTA CRUZ,5290,526,023,75,"18004,93",Directa centro,2025-09-21 04:40:32,21/09/25 06:03:54,S,...,11,12,-75.552368,6.304451,17437,S,0,"46,74",2025-09-21,4
1927,TRANSPORTES ARANJUEZ SANTA CRUZ,6892,525,022,69,"16621,12",Directa centro,2025-09-21 04:37:31,21/09/25 05:52:00,S,...,8,8,-75.549599,6.292368,16630,S,0,"188,77",2025-09-21,4
1928,TRANSPORTES ARANJUEZ SANTA CRUZ,7010,526,023,75,"18004,93",Directa centro,2025-09-21 04:28:55,21/09/25 05:43:30,S,...,3,3,-75.552132,6.304982,17217,S,"68,58","51,16",2025-09-21,4
1929,TRANSPORTES ARANJUEZ SANTA CRUZ,5272,525,022,69,"16621,12",Directa centro,2025-09-21 04:23:23,21/09/25 05:18:48,S,...,3,4,-75.548569,6.291197,16043,S,"176,82","155,53",2025-09-21,4


In [12]:
summary = (
    tra.groupby(["codigo_ruta","idvehiculo", "fechainicio"])["horainicio"]
    .min()
    .groupby(["codigo_ruta","idvehiculo"])
    .std()
)
summary.agg("std")

1.7139219693302052

In [13]:
print("Varianza según la diferencia de horas de salida por bus, de cada ruta:")
print(summary.groupby("codigo_ruta").std().nsmallest())

Varianza según la diferencia de horas de salida por bus, de cada ruta:
codigo_ruta
041 D    0.695222
042      0.798188
022      0.935988
023      1.810211
024      1.934896
Name: horainicio, dtype: float64


In [14]:
# Hallar la ruta con menor varianza en la cantidad de buses
ruta_min = tra.groupby(by="codigo_ruta")["idvehiculo"].nunique().nsmallest()

ruta_min

codigo_ruta
042       19
023       26
022       27
041 D     31
041 I     33
Name: idvehiculo, dtype: int64

# Trabajar con archivo 'Pasajeros'

In [129]:
pas = pd.read_csv("data/01-SecretariadeMovilidad/PASAJEROS_ARANJUEZ_SANTA_CRUZ.csv", encoding="latin-1", sep=";")
pas.columns = pas.columns.str.lower()

In [130]:
columnas_drop = ["empresa", "nombrearchivo"]
pas = pas.drop(columnas_drop, axis=1)

In [131]:
pas

,codigoruta,rutaot,descripcion,fecha,horaregistro,longitud,latitud,velocidad,passuben,pasbajan,consecutivo
0,121275,041 I,ARANJUEZ GUADALUPE,22/09/2025,06:33:39,"-75,546005","6,286640",17,1,0,9
1,121275,041 I,ARANJUEZ GUADALUPE,22/09/2025,06:34:01,"-75,545937","6,287443",14,1,0,11
2,121275,041 I,ARANJUEZ GUADALUPE,22/09/2025,06:34:31,"-75,545860","6,287635",0,4,0,13
3,121275,041 I,ARANJUEZ GUADALUPE,22/09/2025,06:34:52,"-75,546577","6,287738",17,1,0,14
4,121275,041 I,ARANJUEZ GUADALUPE,22/09/2025,06:35:15,"-75,546684","6,287658",6,2,0,16
...,...,...,...,...,...,...,...,...,...,...,...
57898,121265,041 D,ARANJUEZ ANILLO,23/09/2025,20:20:08,"-75,552704","6,284558",6,0,1,216
57899,121265,041 D,ARANJUEZ ANILLO,23/09/2025,20:21:11,"-75,552620","6,286547",0,0,1,219
57900,121265,041 D,ARANJUEZ ANILLO,23/09/2025,20:21:49,"-75,552673","6,287265",12,0,1,221
57901,121265,041 D,ARANJUEZ ANILLO,23/09/2025,20:22:13,"-75,552635","6,287332",0,0,4,223


In [ ]:
# Cast to numeric the lon, lat 
pas["longitud"] = pd.to_numeric(pas["longitud"].str.replace(",", "."))
pas["latitud"] = pd.to_numeric(pas["latitud"].str.replace(",", "."))

In [ ]:
# Cast to datetime
pas["fecha"] = pd.to_datetime(pas["fecha"], format='%d/%m/%Y')
pas["horaregistro"] = pd.to_timedelta(pas["horaregistro"])

In [ ]:
# This form of filter, and groupping gives too chaotic polylines
pas_rutaot1 = pas.groupby(("rutaot")).get_group(("042"))

pas_rutaot1 = pas_rutaot1[pas_rutaot1["fecha"] == pas_rutaot1["fecha"].iat[0]]

ordenada = pas.sort_values(by=["longitud","latitud"], ascending=False)

coords1 = [coord[::-1] for coord in pas_rutaot1[:1000][["longitud", "latitud"]].values.tolist()]

In [ ]:
# Delete point too close
i = 0
while(i < len(coords1)-1):
    if geodesic(coords1[i],coords1[i+1]).meters < 30:
        coords1.pop(i)
    else:
        i += 1

In [196]:
# ---- Filtrar 

pas_rutaot1 = pas.groupby("rutaot").get_group("042")
pas_rutaot1 = pas_rutaot1[pas_rutaot1["fecha"] == pas_rutaot1["fecha"].iat[0]]
pas_rutaot1 = pas_rutaot1.sort_values(by="horaregistro").reset_index(drop=True)

# --- Infer vehicle trips by detecting large jumps ---
MAX_SPEED_KMH = 60  # max realistic bus speed

def compute_trip_id(df, max_speed_kmh=60):
    trip_ids = [0]
    current_trip = 0

    for i in range(1, len(df)):
        prev = df.iloc[i-1]
        curr = df.iloc[i]

        dist_km = geodesic(
            (prev["latitud"], prev["longitud"]),
            (curr["latitud"], curr["longitud"])
        ).km

        time_diff = (curr["horaregistro"] - prev["horaregistro"]).total_seconds() / 3600

        if time_diff > 0:
            speed = dist_km / time_diff
        else:
            speed = float("inf")

        # If speed is physically impossible different vehicle
        if speed > max_speed_kmh:
            current_trip += 1

        trip_ids.append(current_trip)

    df["trip_id"] = trip_ids
    return df

pas_rutaot1 = compute_trip_id(pas_rutaot1)

print(pas_rutaot1["trip_id"].value_counts())

# Pick the longest trip (most GPS pings = most complete route)
best_trip = pas_rutaot1["trip_id"].value_counts().idxmax()
single_trip = pas_rutaot1[pas_rutaot1["trip_id"] == best_trip].reset_index(drop=True)

coords1 = single_trip[["latitud", "longitud"]].values.tolist()

trip_id
491    38
159    16
0      14
1      11
473     8
       ..
206     1
205     1
204     1
203     1
247     1
Name: count, Length: 492, dtype: int64


In [197]:
single_trip= single_trip.sort_values(by=["horaregistro"])
coords1 = single_trip[["latitud", "longitud"]].values.tolist()

In [198]:
# Create a map object centered at a specific location
m = folium.Map(location=coords1[0], zoom_start=13)

for i,coord in enumerate(coords1):
    folium.Marker(
        location = coord,
        icon=folium.Icon(color="blue", icon="info-sign"),
        tooltip=i
    ).add_to(m)

# for coord in coords2:
#     folium.Marker(
#         location = coord,
#         icon=folium.Icon(color="red", icon="info-sign"),
#         tooltip="041 I"
#     ).add_to(m)

folium.PolyLine(locations=coords1).add_to(m)

m.save("mapa.html")

In [190]:
tra["codigo_ruta"].value_counts()

codigo_ruta
024       591
023       347
022       314
041 I     247
042       222
041 D     210
Name: count, dtype: int64

In [ ]:
with open ("data/ruta_coords.json","w") as f:
    json.dump({"coords":coords1}, f, indent=1)

In [ ]:
def get_elevations_batch(coords):
    """coords: list of (lat, lon) tuples"""
    locations = "|".join([f"{lat},{lon}" for lat, lon in coords])
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={locations}"
    response = requests.get(url).json()
    return [r["elevation"] for r in response["results"]]

# Apply to the dataframe
coords = list(zip(single_trip["latitud"], single_trip["longitud"]))
single_trip["elevation"] = get_elevations_batch(coords)

# Compute inclination between consecutive points
single_trip["dist_m"] = [0] + [
    geodesic(coords[i-1], coords[i]).meters 
    for i in range(1, len(coords))
]

single_trip["inclination_deg"] = np.degrees(
    np.arctan2(
        single_trip["elevation"].diff(),
        single_trip["dist_m"]
    )
)